# Standoff2 Player Detection - YOLOv11 Training
Train a YOLOv11n model on the Standoff2 dataset using GPU.
**Estimated time: ~15 minutes**

In [ ]:
# 1. Clone repo
!git clone https://github.com/sprayd3l/standoffai.git
%cd standoffai

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics

In [ ]:
# 3. Extract dataset
import zipfile, os
os.makedirs('data/dataset', exist_ok=True)
with zipfile.ZipFile('data/standoff.zip', 'r') as zf:
    zf.extractall('data/dataset')

# Fix yaml paths
with open('data/dataset/data.yaml', 'r') as f:
    y = f.read()
y = y.replace('../train/images', 'train/images')
y = y.replace('../valid/images', 'valid/images')
y = y.replace('../test/images', 'test/images')
with open('data/dataset/data.yaml', 'w') as f:
    f.write(y)
print('Dataset ready:', len(os.listdir('data/dataset/train/images')), 'train images')

In [ ]:
# 4. Train YOLOv11n (GPU)
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(
    data='data/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    project='runs',
    name='standoff2',
    exist_ok=True,
    verbose=True,
)

In [ ]:
# 5. Export models
best = YOLO('runs/standoff2/weights/best.pt')
best.export(format='tflite', imgsz=640)
best.export(format='onnx', imgsz=640)
print('Exported!')

import os
for f in ['best.pt', 'best.tflite', 'best.onnx']:
    path = f'runs/standoff2/weights/{f}'
    if os.path.exists(path):
        print(f'  {f}: {os.path.getsize(path)/1024/1024:.1f} MB')

In [ ]:
# 6. Download models
from google.colab import files
files.download('runs/standoff2/weights/best.tflite')